[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrobledob/AI_in_Plant_Pathology_Fall_2026/blob/main/notebooks/01_feature_matrix.ipynb)

# Notebook 1 — From field data to a feature matrix

**Supervised machine learning for plant pathologists**

Bacterial spot of tomato (*Xanthomonas perforans*), Florida. 480 field-seasons across
8 farms and 10 seasons, plus 5,760 plot-weeks.

By the end of this notebook you will have:

1. decided what one row of the data means,
2. built a feature matrix `X` and a target `t`,
3. wrapped preprocessing into a `Pipeline` so it cannot leak,
4. measured how much a careless train/test split flatters a model.

The data is synthetic, generated from a documented process in `truth.json`. That means
we can check, at the end of Notebook 2, exactly what the model should have recovered.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.width', 120)
plt.rcParams['figure.figsize'] = (8, 4)

import os

# Data lives next to this notebook when you clone the repo, and on GitHub when you
# open it in Colab. This finds it either way.
REPO = ('https://raw.githubusercontent.com/'
        'jrobledob/AI_in_Plant_Pathology_Fall_2026/main')
DATA = 'data' if os.path.isdir('data') else f'{REPO}/notebooks/data'
print('reading data from:', DATA)

---
## 1. Load the data

In [ ]:
season = pd.read_csv(f'{DATA}/bacterial_spot_season.csv')
weekly = pd.read_csv(f'{DATA}/bacterial_spot_weekly.csv')

season.shape, weekly.shape

In [ ]:
season.head()

### What are the columns?

* **Identifiers**: `year`, `site`, `county`, `plot`
* **Features**: weather (`wetness_hours`, `mean_temp_c`, `max_temp_c`, `rain_mm`),
  host (`cultivar`), inoculum (`inoculum_log`), management (`transplant_source`),
  and two others we will come back to (`soil_ph`, `dist_to_road_m`)
* **Targets**: `final_severity` (% leaf area) and `audpc`

In [ ]:
season.dtypes

---
## 2. What is one row?

This is the decision the slides called the most consequential one in the workflow.
The same field data can be organised three ways, and each gives a different `n`.

In [ ]:
print('field-seasons :', len(season))
print('plot-weeks    :', len(weekly))
print()
print('plots per site-year:', season.groupby(['site', 'year']).size().unique())
print('weeks per plot     :', weekly.groupby('plot').size().unique())

Twelve weekly rows share a single plot. They share its cultivar, its inoculum level and
its microclimate. They are **not** twelve independent observations.

Keep that in mind — we will measure what it costs at the end of this notebook.

---
## 3. Look at the target before you model it

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].hist(season.final_severity, bins=25, color='#3677A5', edgecolor='white')
axes[0].set_xlabel('Final severity (%)')
axes[0].set_ylabel('Field-seasons')

season.groupby('year').final_severity.mean().plot(kind='bar', ax=axes[1],
                                                  color='#FF4A00')
axes[1].set_ylabel('Mean severity (%)')
axes[1].set_xlabel('')
plt.tight_layout()

Two things worth noticing, and both matter later:

1. Severity is a **bounded proportion**. It cannot go below 0 or above 100, and the
   distribution crowds near both ends. A linear model does not know that.
2. Season means swing from roughly 30% to nearly 70%. That is a **year effect** —
   whole seasons are good or bad together. It is the reason a random split will lie
   to us in section 6.

In [ ]:
# Exercise: how much does severity vary between sites, compared to between years?
print('SD of site means:', round(season.groupby('site').final_severity.mean().std(), 2))
print('SD of year means:', round(season.groupby('year').final_severity.mean().std(), 2))

---
## 4. Missing values

Real rating data has holes. Ours has a few, on purpose.

In [ ]:
season.isna().sum()[lambda s: s > 0]

We will **not** drop these rows and we will **not** fill them by hand. The imputation
goes inside the pipeline, so the fill value is computed from training data only.

---
## 5. Build X and t

In [ ]:
NUM = ['wetness_hours', 'mean_temp_c', 'max_temp_c', 'rain_mm',
       'inoculum_log', 'soil_ph', 'dist_to_road_m']
CAT = ['cultivar', 'transplant_source']

X = season[NUM + CAT]
t = season['final_severity']

X.shape, t.shape

Notice what is **not** in `X`: `year`, `site`, `county`, `plot` and `audpc`.

* The identifiers are how we will *split* the data, not what we predict from.
* `audpc` is derived from `final_severity`. Using it as a feature would be leakage of
  the most direct kind — predicting the target from the target.

---
## 6. Preprocessing, inside a pipeline

Numeric columns need imputing and scaling. Categorical columns need one-hot encoding.
Both must be fitted on training data only, which is exactly what a `Pipeline` inside
cross-validation guarantees.

In [ ]:
num_steps = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])

pre = ColumnTransformer([
    ('num', num_steps, NUM),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), CAT),
])

model = Pipeline([('pre', pre), ('lin', LinearRegression())])
model

In [ ]:
# What do the columns look like after preprocessing?
Xp = pre.fit_transform(X)
feature_names = NUM + list(pre.named_transformers_['cat'].get_feature_names_out(CAT))

pd.DataFrame(Xp, columns=feature_names).head()

`cultivar` has five levels and became four columns. `FL-8000` is the reference: all
zeros. Every cultivar weight will be read against it.

---
## 7. The split exercise

This is the point of the notebook.

We will score the **same model** three ways:

* **random** 10-fold — rows shuffled, ignoring structure
* **leave-one-site-out** — whole farms held out
* **leave-one-year-out** — whole seasons held out

`GroupKFold` keeps every row of a group in the same fold.

In [ ]:
def rmse_cv(cv, groups=None):
    scores = cross_val_score(model, X, t, cv=cv, groups=groups,
                             scoring='neg_root_mean_squared_error')
    return -scores.mean()

random_rmse = rmse_cv(KFold(10, shuffle=True, random_state=0))
site_rmse   = rmse_cv(GroupKFold(8), groups=season.site)
year_rmse   = rmse_cv(GroupKFold(10), groups=season.year)

print(f'random 10-fold      RMSE {random_rmse:5.2f}')
print(f'leave-one-site-out  RMSE {site_rmse:5.2f}')
print(f'leave-one-year-out  RMSE {year_rmse:5.2f}')

In [ ]:
# And the baseline: predict the overall mean for every field-season
baseline = np.sqrt(((t - t.mean()) ** 2).mean())
print(f'predicting the mean RMSE {baseline:5.2f}')

### Discuss with the person next to you

* Which of the three numbers would you put in a manuscript?
* The gap here is modest. Why is it not larger for **this** model?
* What would make it larger?

Hold that last question — the next cell answers it.

---
## 8. Now make the leakage bite

A linear model with seven weather columns cannot memorise much. A flexible model can.

We switch to the **weekly** data, where twelve rows share a plot, and to a random
forest, which is free to memorise whatever identifies a plot.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

WNUM = ['wet_hours_week', 'night_temp_c', 'rain_mm_week',
        'inoculum_log', 'days_after_transplant']
Xw = weekly[WNUM + ['cultivar']]
tw = weekly['infection_event']

# Trees do not need a reference level dropped, so we keep all five cultivar columns
pre_w = ColumnTransformer([
    ('num', StandardScaler(), WNUM),
    ('cat', OneHotEncoder(sparse_output=False), ['cultivar']),
])

rf = Pipeline([('pre', pre_w),
               ('rf', RandomForestClassifier(n_estimators=200, min_samples_leaf=2,
                                             random_state=0))])

In [ ]:
def auc_cv(estimator, cv, groups=None):
    return cross_val_score(estimator, Xw, tw, cv=cv, groups=groups,
                           scoring='roc_auc').mean()

print('random forest')
print(f'  random 10-fold     AUC {auc_cv(rf, KFold(10, shuffle=True, random_state=0)):.3f}')
print(f'  leave-one-plot-out AUC {auc_cv(rf, GroupKFold(10), weekly["plot"]):.3f}')
print(f'  leave-one-year-out AUC {auc_cv(rf, GroupKFold(10), weekly["year"]):.3f}')

Roughly four points of AUC, and every one of them is the model recognising plots and
seasons it had already seen.

The random-split number is the one that would have been published.

---
## 9. What to take from this notebook

* One row is a decision, not a given. Ours is a field-season for severity and a
  plot-week for infection events.
* Preprocessing belongs inside the pipeline, so that scaling and imputation never see
  the test fold.
* `GroupKFold` on year, site or plot is the honest default for field data.
* The more flexible the model, the more a careless split flatters it.

**Next**: Notebook 2 fits the linear model properly and reads its weights.